In [66]:
%load_ext autoreload
%autoreload 2
%reset -f

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [67]:
from pathlib import Path
import os
from os.path import join
import sys

# Find KPIHub root (directory containing lib/tables), then import lib.* as a package.
_here = Path.cwd().resolve()
_root = next((p for p in [_here, *list(_here.parents)[:12]] if (p / "lib" / "tables").is_dir()), None)
if _root is None:
    raise FileNotFoundError(
        f"Could not find KPIHub root (folder containing lib/tables). cwd={Path.cwd()!r}"
    )
sys.path.insert(0, str(_root))
os.chdir(_root)

from locallib.picarrodb import *
from locallib.slack import *
from locallib.etl import Loggers
from locallib.pandas import *

from lib.tables.IngesterTables import *
from lib.handlers.CustomerHandler import *
from lib.kpi_processor.KPIReport import *
from lib.config import *
from lib.KPIHubConnection import *
from lib.query.bank import *

from datetime import date
from datetime import timedelta

In [68]:
class KPIPOR_v2(KPIPOR):
    def __init__(self, customer_name, aggregator = {}, period_dict = {'Week': 'ReportWeek'}):
        super().__init__(customer_name, aggregator, period_dict)

    def process_data(self, on='ReportId'):
        PORData = Query(query = f"SELECT * FROM KPI_POR WHERE CustomerId = '{self.customer_id}'").execute(KPIHub_Conn)
        PORValue = PORData['Value'].values[0]
        # Attach POR value to agg_df as a new column named 'POR'
        if PORValue == 0:
            raise ValueError(f"POR not found for {customer_name}")
        else:
            reports = POR.data[KPI_ReportSummary]
            # -- Fix: Filter reports that are within ANY PORData period, accounting for multiple periods --
            mask = reports.apply(
                lambda row: any(
                    pd.to_datetime(row['ReportDate']) >= pd.to_datetime(por_row['StartingDate']) and
                    pd.to_datetime(row['ReportDate']) <= pd.to_datetime(por_row['EndingDate'])
                    for _, por_row in PORData.iterrows()
                ),
                axis=1
            )
            PORReports = reports[mask]
            report_years = []
            report_weeks = []
            values = []

            for _, row in PORData.iterrows():
                value = row['Value']
                start_date = pd.to_datetime(row['StartingDate'])
                end_date = pd.to_datetime(row['EndingDate'])

                start_year = start_date.year
                end_year = end_date.year

                for year in range(start_year, end_year + 1):
                    if year == start_year:
                        first_week_date = start_date
                    else:
                        first_week_date = pd.to_datetime(f"{year}-01-01")

                    if year == end_year:
                        last_week_date = end_date
                    else:
                        last_week_date = pd.to_datetime(f"{year}-12-31")

                    week_date = first_week_date
                    week_date = week_date - pd.Timedelta(days=week_date.weekday())  # previous Monday

                    while week_date <= last_week_date:
                        iso_calendar = week_date.isocalendar()
                        week_number = iso_calendar.week
                        year_number = iso_calendar.year
                        if year_number == year:
                            report_years.append(year)
                            report_weeks.append(week_number)
                            values.append(value)
                        week_date = week_date + pd.Timedelta(weeks=1)

                por_summary_df = pd.DataFrame({
                    'ReportYear': report_years,
                    'ReportWeek': report_weeks,
                    'POR': values
                }).set_index(['ReportYear', 'ReportWeek'])

                # Perform the aggregation and assign to a new DataFrame to avoid SettingWithCopyWarning
            agg_df = PORReports.groupby(POR.aggregator).agg({
                'AssetCoveredLengthKm': 'sum',
            })

            agg_df['CumulativeAssetCoveredLengthKm'] = agg_df.groupby('ReportYear')['AssetCoveredLengthKm'].cumsum()

            agg_df = agg_df.join(por_summary_df, how="left")
            agg_df['CurrentCompletion'] = agg_df['CumulativeAssetCoveredLengthKm'] / agg_df['POR']
            agg_df['CurrentCompletion'] = (100*(agg_df['CurrentCompletion'])).round(2)
            self.data['output'] = agg_df

            self.melter()


In [69]:
POR = KPIPOR_v2('Cadent')
POR.query_table()
POR.process_data()
POR.push_data()